# NOVA 01 — MOD-01 Obstacle Detection (YOLOv8n)
**Attach datasets** (Add Data, right sidebar):
- `jhontroya/dectectra-dataset`
- `kushagrapandya/visdrone-dataset`

**Accelerator:** GPU T4 x2 or P100. Full training ~6-9h — fits one Kaggle session (12h limit). Enable *Persistence: Files* to survive restarts.

In [1]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

Cloning into '/kaggle/working/nova-ml'...


GPU OK: Tesla T4
Bootstrap OK — repo cloned, HF token loaded.


In [2]:
!pip install -q ultralytics onnx2tf onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1

In [3]:
# Resolve the ACTUAL mount paths — Kaggle sometimes mounts datasets
# nested as /kaggle/input/datasets/<owner>/<slug>, so search 3 levels.
import glob
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
DETECTRA = next(p for p in inputs if 'dect' in p.split('/')[-1].lower())
VISDRONE = next((p for p in inputs if 'visdrone' in p.split('/')[-1].lower()), None)
print('Detectra:', DETECTRA)
print('VisDrone:', VISDRONE)
!find {DETECTRA} -maxdepth 3 -type d | head -20

Detectra: /kaggle/input/datasets/jhontroya/dectectra-dataset
VisDrone: /kaggle/input/datasets/kushagrapandya/visdrone-dataset
/kaggle/input/datasets/jhontroya/dectectra-dataset
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/images
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/images/val
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/images/test
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/images/train
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/labes
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/labes/val
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/labes/test
/kaggle/input/datasets/jhontroya/dectectra-dataset/train_data/labes/train


In [4]:
# Free disk first: /kaggle/working is only 19.5 GB and stale copies from
# earlier runs fill it. Images are SYMLINKED (not copied) into the merge,
# so the combined dataset itself costs ~100 MB of labels only.
!rm -rf /kaggle/working/datasets
!df -h /kaggle/working | tail -1
# Detectra-only (v1.0.0 lesson: VisDrone's aerial drone perspective is
# wrong for a chest-mounted camera — its tiny objects at 320px dragged
# mAP50 down to 10%. Pedestrian-height Detectra alone is the right data).
# GENERATES the training YAML with correct nc/names; aborts if 0 images.
!python scripts/prepare_obstacle_dataset.py \
    --detectra {DETECTRA} \
    --out /kaggle/working/datasets/obstacle_combined \
    --yaml-out /kaggle/working/obstacle_data.yaml
!head -40 /kaggle/working/obstacle_data.yaml

/dev/loop1       20G  544K   20G   1% /kaggle/working
  (detectra: skipped 52 images with no matching label file)
Detectra: {'train': 7548, 'val': 1900}
VisDrone: {'train': 6471, 'val': 548} (index remap: {0: 77, 1: 77, 2: 78, 3: 79, 4: 80, 5: 81, 6: 82, 7: 83, 8: 84, 9: 85})

Combined: 14019 train / 2448 val images, 86 classes
Training config written to /kaggle/working/obstacle_data.yaml — pass it as --data to train_obstacle.py
path: /kaggle/working/datasets/obstacle_combined
train: images/train
val: images/val
nc: 86
names:
  0: class_0
  1: class_1
  2: class_2
  3: class_3
  4: class_4
  5: class_5
  6: class_6
  7: class_7
  8: class_8
  9: class_9
  10: class_10
  11: class_11
  12: class_12
  13: class_13
  14: class_14
  15: class_15
  16: class_16
  17: class_17
  18: class_18
  19: class_19
  20: class_20
  21: class_21
  22: class_22
  23: class_23
  24: class_24
  25: class_25
  26: class_26
  27: class_27
  28: class_28
  29: class_29
  30: class_30
  31: class_31
  32: cl

In [5]:
# Full training + TFLite INT8 export (Ultralytics native export).
# STOP if the previous cell errored — do not burn GPU hours on 0 images.
# Fast profile: 40 epochs + early stopping (patience 10) + batch 64 +
# 4 dataloader workers ≈ 2-3h on a T4 and typically lands within ~1-2
# mAP points of a 100-epoch run (YOLOv8n starts COCO-pretrained).
# Add --fraction 0.5 to halve it again if you're in a real hurry.
# If a session dies mid-run, re-run this cell with --resume.
!python scripts/train_obstacle.py --data /kaggle/working/obstacle_data.yaml \
    --epochs 60 --imgsz 320 --batch 64 --workers 4 --patience 15

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/obstacle_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv

In [6]:
# Publish to HuggingFace: unixio/nova-obstacle-detection
import glob
candidates = glob.glob('/kaggle/working/runs/obstacle_student/weights/**/*_int8.tflite',
                       recursive=True)
print('TFLite candidates:', candidates)
assert candidates, 'No INT8 TFLite found — training/export must succeed first.'
tflite_path = candidates[0]
best_pt = '/kaggle/working/runs/obstacle_student/weights/best.pt'
!python scripts/push_to_huggingface.py --module MOD-01 \
    --pytorch {best_pt} --tflite {tflite_path} \
    --eval-json /kaggle/working/evaluation/obstacle_results.json --version 1.1.0

TFLite candidates: ['/kaggle/working/runs/obstacle_student/weights/best_int8.tflite']
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  .../weights/best_int8.tflite:  16%|██▏           |  560kB / 3.52MB            

Processing Files (0 / 1)      :  16%|██▏           |  560kB / 3.52MB,   ???B/s  
New Data Upload               :  16%|██▏           |  560kB / 3.52MB,   ???B/s  

Processing Files (0 / 1)      :  95%|█████████████▎| 3.36MB / 3.52MB, 14.0MB/s  
New Data Upload               :  95%|█████████████▎| 3.36MB / 3.52MB, 14.0MB/s  

Processing Files (1 / 1)      : 100%|██████████████| 3.52MB / 3.52MB, 7.40MB/s  
New Data Upload               : 100%|██████████████| 3.52MB / 3.52MB, 7.40MB/s  

  .../weights/best_int8.tflite: 100%|██████████████| 3.52MB / 3.52MB            

Processing Files (1 / 1)      : 100%|██████████████| 3.52MB / 3.52MB, 4.93MB/s  
New Data Upload  